**Problem 4**: So far, we've assumed that the wheels can instantly produce whatever velocity or torque our controller commands. In reality, the controller can only command the motor voltage (typically via PWM), while the motor's current, torque, and speed evolve according to its electrical and mechanical dynamics. To design a controller that works on a real robot, we first need a model of the motor itself.

First, we model our DC motor. Applying Kirchoff's law around the armature yields:
$$
V = Ri + L \dfrac{di}{dt} + K_e \omega
$$
where:
- $V$ is voltage
- $R$ is the resistance of the armature
- $L$ is the inductance of the armature
- $K_e$ is the back electromagnetic force (EMF) constant

This is the electrodynamics of the system. But we still need to connect it to the mechanical model:
$$
I \dot{\omega} = \tau - b \omega - \tau_{load}
$$
In fact, we also have:
$$ e = K_e \omega$$
where $e$ is the electromotive force, and:
$$ \tau = K_t i $$
where $K_i$ is the torque constant. What these two equations say is basically: the electromotive force generated by a DC motor is proportional to the angular velocity, and the torque of the motor is proportional to the current going through it. For an ideal motor with no losses, we also have:
$$ P_{elec} = P_{mech}$$
by conservation of energy. But $P_{elec} = ei$, and $P_{mech} = \tau \omega$ so it means that for an ideal motor described in SI units:
$$ K_e = K_t$$
Now there are two ways for us to describe and analyse the system. We'll use both:

**1. State space representation**

A state is the minimum information we need to determine the system's future behaviour. Typically, we write:
$$ \dot{\mathbf{x}} = \mathbf{Ax} + \mathbf{Bu}$$
where $x$ is the state vector, $u$ is the control input, and $A$, $B$ are coefficient matrices. So, $\dot{x}$ tells you how the system is changing at every step. For programming, we can discretize the system using the equation:
$$\mathbf{{x_{k+1}}} = \mathbf{Ax_k} + \mathbf{Bu_k}$$
For our DC motor model, we can choose:
$$ \mathbf{x} = \begin{bmatrix} i \\ \omega \end{bmatrix} \hspace{40pt} \mathbf{u} = V$$
Then our dynamics would be:
$$\dot{i} = \dfrac{1}{L} (V - Ri - K_e \omega)$$
$$\dot{\omega} = \dfrac{1}{I} ( K_t i - b \omega - \tau_{load})$$
In matrix form, that is:
$$
\begin{bmatrix} \dot{i} \\ \dot{\omega} \end{bmatrix} = \begin{bmatrix} -\dfrac{R}{L} & -\dfrac{K_e}{L} \\ \dfrac{K_t}{I} & -\dfrac{b}{I} \end{bmatrix} \begin{bmatrix} i \\ \omega \end{bmatrix} + \begin{bmatrix} \dfrac{1}{L} \\ 0 \end{bmatrix} V
$$
But this is quite theoretical and practically uneccessary. In fact, we can drop the inductance term. Inductance resists the change in current, and often times, the electrical dynamics settle far faster than the mechanical dynamics. The electrical time constant is $\tau = \dfrac{L}{R}$. Because both inductance and resistance in an armature are small, by dropping that term, we arrive at:
$$
i = \dfrac{V - K_e \omega}{R}
$$
From here, controlling torque is simple. Plugging in any value of $V$ gives us current, and from there we get torque. We get a simple understanding of how the system behaves over time, locally.

Basically, we arrived at the fact that voltage does not directly control speed. It first changes current, which produces torque, which then accelerates the motor. To analyse how the output change directly from information of the input, we can use **transfer functions**.

**2. Transfer functions**

A transfer function tells you exactly how an output of a system respond, given the input. Normally it's written as:
$$ G(s) = \dfrac{\text{output(s)}}{\text{input(s)}} $$
The $s$ here means we're working in the complex frequency domain. We take the Laplace transform of everything to better analyse the input signals. It is easier than using differential equations.

Now let's find the transfer function for our DC motor. We want to find how the angular velocity changes as the voltage change, so we define:
$$ G(s) = \dfrac{\Omega(s)}{V(s)}$$

Knowing the dynamics from above, we will take the Laplace transform of their ODEs. What this makes is that it turns derivatives and integral behaviour into algebraic multiplication. We arrive at:
$$ V(s) = RI(s) + LsI(s) + K_e \Omega(s)$$
$$ Is\Omega(s) + b\Omega(s) = K_t I(s)$$
The rest is simple algebra. We solve for current:
$$ I(s) = \dfrac{Is + b}{K_t} \Omega(s)$$
Plugging back in to the voltage equation yields:
$$ V(s) = \Omega(s) \left[ \dfrac{(R+Ls)(Is + b)}{K_t} \right]$$
Dividing both sides and assuming $L \approx 0$ yields the transfer function:
$$ G(s) = \dfrac{\Omega(s)}{V(s)} = \dfrac{K_t}{RIs + Rb + K_t K_e}$$
Because voltage is a signal, we can plug in a signal $V(s)$, take the inverse Laplace transform of the system to move back to the time domain and see exactly how the system behaves over time. Instead of a local behaviour (this speed at this voltage), we now know the global behaviour of the system.